In [4]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from flax import serialization as sz

# If you really want seaborn for styling (optional):
# import seaborn as sns; sns.set()

# Your model module (adjust the path/name if needed)
from model import (
    MiniGPJax,
    # kernel / mean / likelihood classes
    RBF, Matern32, GaussianLikelihood, ZeroMean, GPParams, Standardizer,
    # state (de)serializers defined in your file
    _kernel_to_state, _kernel_from_state,
    _lik_to_state, _lik_from_state,
    _zeromean_to_state, _zeromean_from_state,
    _gpparams_to_state, _gpparams_from_state,
    _std_to_state, _std_from_state,
)
from model import MiniGPJax, _register_all_serializers_once
_register_all_serializers_once()

## Create a GP code surrogate and then save it

### Making sure it works since I will write the bayesian inference code here

In [2]:
TRAIN_CSV = "Completed_train_set_10_30_25.csv"
TEST_CSV  = "Completed_test_set_10_30_25.csv"
CKPT_DIR  = "ckpts/gp_run_001"
dim = 5
gp = MiniGPJax(
    dim=dim,
    kernel="rbf",            # or "matern32" I dont have the others
    fixed_noise=None,        # consider None to learn noise for better generalization
    standardize_x=True,
    standardize_y=True,
    log_y=True,              # ensures: log -> standardize (train stats) -> fit
    jitter=1e-6,
)
gp.load(TRAIN_CSV, TEST_CSV)
gp.fit(lbfgs_max_iter=2, lbfgs_tol=1e-7, num_restarts=1, seed=42)

#

Platform 'METAL' is experimental and not all JAX functionality may be correctly supported!
W0000 00:00:1762609048.415048 44518874 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1762609048.426773 44518874 service.cc:145] XLA service 0x6000015c1600 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1762609048.426786 44518874 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1762609048.428006 44518874 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1762609048.428017 44518874 mps_client.cc:384] XLA backend will use up to 22906109952 bytes on device 0 for SimpleAllocator.


Metal device set to: Apple M2 Max
[ok] Loaded train: X=(1024, 5), y=(1024,) (log→std)
[ok] Loaded test:  X=(256, 5), y=(256,) (log→std)
[restart 0] nLML = 150.665283
[ok] Best nLML = 150.665283
[fit] amp=0.5988 | noise_var=0.006656 | ell=[1.201 2.446 2.233 1.242 2.251]
[fit] train RMSE(z)=0.0977 | R²(z)=0.9905  (z = standardized)


In [5]:
_register_all_serializers_once()
gp.save_checkpoint(CKPT_DIR)


[save] Wrote checkpoint to: /Users/akshayjacobthomas/Documents/GitHub/BayesianFLow_calibration/ckpts/gp_run_001


In [6]:
# Cell 4: load + rebuild posterior
_register_all_serializers_once()
gp2 = MiniGPJax.ready_from_checkpoint(CKPT_DIR, train_csv=TRAIN_CSV, test_csv=TEST_CSV, rebuild=True)


TypeError: _gpparams_from_state() takes 1 positional argument but 2 were given